In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🤖 Machine Learning Operations (MLOps) Guide

**Building production ML systems with versioning, monitoring, and automation**

This guide covers:
- ML model versioning and registry
- Feature store management
- Model training orchestration
- Model serving and inference
- Model monitoring and drift detection
- A/B testing for models
- Retraining pipelines
- Cost optimization for ML
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Model Registry & Versioning

### MLflow for Model Management
```python
import mlflow
from sklearn.ensemble import RandomForestClassifier

mlflow.set_experiment("aria-classifier")

with mlflow.start_run(run_name="baseline_v1"):
    # Train model
    X_train, X_test, y_train, y_test = load_data()
    model = RandomForestClassifier(n_estimators=100)
    model.fit(X_train, y_train)
    
    # Calculate metrics
    accuracy = model.score(X_test, y_test)
    precision = precision_score(y_test, model.predict(X_test))
    recall = recall_score(y_test, model.predict(X_test))
    
    # Log metrics
    mlflow.log_metrics({
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall
    })
    
    # Log model
    mlflow.sklearn.log_model(
        model,
        artifact_path="model"
    )
    
    # Log parameters
    mlflow.log_params({
        'n_estimators': 100,
        'max_depth': 10
    })
    
    # Register model
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"
    registered = mlflow.register_model(
        model_uri=model_uri,
        name="aria-classifier"
    )

# Load and serve model
model = mlflow.pyfunc.load_model("models:/aria-classifier/production")
predictions = model.predict(X_test)
```

### Model Versioning Strategy
```python
class ModelRegistry:
    def __init__(self):
        self.client = mlflow.tracking.MlflowClient()
    
    async def promote_model(self, run_id: str, stage: str):
        """Promote model through stages"""
        
        # Stages: None → Staging → Production → Archived
        model_version = self.client.get_model_version_by_alias(
            name="aria-classifier",
            alias="production"
        )
        
        # Validation before promotion
        metrics = self.client.get_run(run_id).data.metrics
        
        if metrics['accuracy'] < 0.90:
            raise ValueError(f"Accuracy too low: {metrics['accuracy']}")
        
        # Promote
        self.client.set_model_version_tag(
            name="aria-classifier",
            version=run_id,
            key="stage",
            value=stage
        )
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Feature Store

### Feature Management
```python
from feast import FeatureStore, FeatureView, Entity

# Define feature store
fs = FeatureStore(repo_path="./feature_repo")

# Define entity
user = Entity(name="user_id", description="User ID")

# Define feature view
user_features = FeatureView(
    name="user_features",
    entities=[user],
    features=[
        Feature(name="age", dtype=ValueType.INT64),
        Feature(name="city", dtype=ValueType.STRING),
        Feature(name="total_purchases", dtype=ValueType.FLOAT),
    ],
    input=PostgreSQLSource(
        query="SELECT user_id, age, city, total_purchases FROM users"
    ),
    ttl=timedelta(hours=1),
)

# Register features
fs.apply([user_features])

# Get features for training
training_df = fs.get_historical_features(
    entity_df=pd.DataFrame({'user_id': [1, 2, 3, ...]}),
    features=['user_features:age', 'user_features:city']
).to_df()

# Get features for serving
feature_vector = fs.get_online_features(
    features=['user_features:age', 'user_features:city'],
    entity_rows=[{'user_id': 123}]
).to_dict()
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 3. Model Training Pipeline

### Automated Training with Orchestration
```python
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

default_args = {
    'owner': 'ml-team',
    'retries': 2,
    'retry_delay': timedelta(minutes=5),
}

dag = DAG(
    'model_training_pipeline',
    default_args=default_args,
    schedule_interval='@daily',
)

def prepare_training_data():
    """Extract and prepare training data"""
    X_train, y_train = load_training_data()
    return X_train, y_train

def train_model(X_train, y_train):
    """Train ML model"""
    model = RandomForestClassifier(n_estimators=100)
    model.fit(X_train, y_train)
    
    # Save checkpoint
    mlflow.sklearn.log_model(model, "model")
    
    return model

def evaluate_model(model):
    """Evaluate on test set"""
    X_test, y_test = load_test_data()
    accuracy = model.score(X_test, y_test)
    
    if accuracy < 0.90:
        raise ValueError(f"Model accuracy too low: {accuracy}")
    
    return accuracy

def deploy_model(model):
    """Deploy to production"""
    # Register in model registry
    mlflow.register_model(model, "production")
    
    # Deploy to serving infrastructure
    serve_model(model)

# Define DAG tasks
prepare = PythonOperator(task_id='prepare_data', python_callable=prepare_training_data)
train = PythonOperator(task_id='train', python_callable=train_model)
evaluate = PythonOperator(task_id='evaluate', python_callable=evaluate_model)
deploy = PythonOperator(task_id='deploy', python_callable=deploy_model)

prepare >> train >> evaluate >> deploy
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 4. Model Monitoring & Drift Detection

### Data Drift Detection
```python
from sklearn.ensemble import IsolationForest

class DriftDetector:
    def __init__(self, reference_data, threshold=0.05):
        self.reference_data = reference_data
        self.threshold = threshold
        self.drift_detector = IsolationForest(contamination=threshold)
        self.drift_detector.fit(reference_data)
    
    def detect_drift(self, current_data):
        """Detect if current data distribution differs from reference"""
        
        anomaly_scores = self.drift_detector.decision_function(current_data)
        
        # Average anomaly score
        drift_score = anomaly_scores.mean()
        
        if drift_score > self.threshold:
            logger.warning(f"Data drift detected: {drift_score}")
            alert_team("Data drift detected, consider retraining")
            return True
        
        return False

# Monitor prediction drift
class PredictionMonitor:
    def __init__(self):
        self.predictions = []
        self.actuals = []
    
    async def log_prediction(self, input_data, prediction, actual=None):
        """Log predictions for monitoring"""
        
        self.predictions.append(prediction)
        if actual:
            self.actuals.append(actual)
        
        # Evaluate drift every 1000 predictions
        if len(self.predictions) % 1000 == 0:
            accuracy = (
                sum(p == a for p, a in zip(self.predictions, self.actuals)) /
                len(self.actuals)
            )
            
            if accuracy < 0.85:  # Threshold
                logger.error(f"Model accuracy dropped to {accuracy}")
                trigger_retraining()
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 5. A/B Testing for Models

```python
class ModelABTest:
    def __init__(self, control_model, treatment_model, split_ratio=0.5):
        self.control = control_model
        self.treatment = treatment_model
        self.split_ratio = split_ratio
        self.results = {'control': [], 'treatment': []}
    
    async def predict_with_assignment(self, user_id: str, input_data):
        """Assign user to control or treatment"""
        
        # Consistent assignment using user_id hash
        is_treatment = hash(user_id) % 100 < (self.split_ratio * 100)
        
        if is_treatment:
            prediction = self.treatment.predict(input_data)
            self.results['treatment'].append(prediction)
            return prediction, 'treatment'
        else:
            prediction = self.control.predict(input_data)
            self.results['control'].append(prediction)
            return prediction, 'control'
    
    def get_statistics(self):
        """Compare models statistically"""
        
        control_acc = calculate_accuracy(self.results['control'])
        treatment_acc = calculate_accuracy(self.results['treatment'])
        
        # T-test for significance
        t_stat, p_value = ttest_ind(
            self.results['control'],
            self.results['treatment']
        )
        
        return {
            'control_accuracy': control_acc,
            'treatment_accuracy': treatment_acc,
            'improvement': treatment_acc - control_acc,
            'p_value': p_value,
            'significant': p_value < 0.05
        }
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 6. MLOps Checklist

- [ ] Model versioning strategy defined
- [ ] Feature store implemented
- [ ] Training pipeline automated
- [ ] Model registry set up
- [ ] Monitoring dashboards created
- [ ] Drift detection implemented
- [ ] A/B testing framework ready
- [ ] Retraining triggers defined
- [ ] Model rollback procedures documented
- [ ] Cost optimization for serving

**Key Metrics**: Model accuracy, latency, drift score, cost per prediction
</MySCode.Cell>
```